Install & Authenticate Kaggle, Then Download Datasets

In [ ]:
import os
from google.colab import userdata

# Load Kaggle credentials from Colab Secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle scikit-learn seaborn

# Create raw data folders
!mkdir -p raw_data/philmedic raw_data/plantvillage

# 1. Download Philippine Medicinal Plant Leaf Dataset (4,922 images)
!kaggle datasets download -d hartanto/philippine-medicinal-plant-leaf-dataset -p raw_data/philmedic --unzip

# 2. Download PlantVillage Dataset (for Non-Medicinal Unknown class & OOD testing)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p raw_data/plantvillage --unzip

print("Downloads complete! Inspecting folder structures...")
for root, dirs, files in os.walk("raw_data"):
    if len(files) > 50:
        print(f"{root}: {len(files)} images")

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:20<00:00, 105MB/s]

Downloads complete! Inspecting folder structures...
raw_data/plantvillage/plantvillage dataset/grayscale/Tomato___Spider_mites Two-spotted_spider_mite: 1676 images
raw_data/plantvillage/plantvillage dataset/grayscale/Potato___Early_blight: 1000 images
raw_data/plantvillage/plantvillage dataset/grayscale/Tomato___Septoria_leaf_spot: 1771 images
raw_data/plantvillage/plantvillage dataset/grayscale/Apple___Black_rot: 621 images
raw_data/plantvillage/plantvillage dataset/grayscale/Strawberry___Leaf_scorch: 1109 images
raw_data/plantvillage/plantvillage dataset/grayscale/Peach___healthy: 360 images
raw_data/plantvillage/plantvillage dataset/grayscale/Squash___Powdery_mildew: 1835 images
raw_data/plantvillage/plantvillage dataset/graysca

In [ ]:
import os

# Download the Philippine Medicinal Plant Leaf Dataset using the exact Kaggle slug
!kaggle datasets download -d jaydepaolomirandilla/philippine-medicinal-plant-leaf-dataset -p raw_data/philmedic --unzip

print("Checking extracted philmedic folders...")
for root, dirs, files in os.walk("raw_data/philmedic"):
    if len(files) > 0:
        print(f"{root}: {len(files)} images")

Dataset URL: https://www.kaggle.com/datasets/jaydepaolomirandilla/philippine-medicinal-plant-leaf-dataset
License(s): other
100% 7.45G/7.45G [01:20<00:00, 99.6MB/s]

Checking extracted philmedic folders...
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/18Mentha cordifolia Opiz(MCO): 100 images
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/29Premna odorata(PO): 120 images
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/34Impatiens balsamina(IB): 130 images
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/32Mangifera indica(MI): 110 images
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/2Psidium guajava(PG): 110 images
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/14Euphorbia hirta(EH): 128 images
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/35Arachis hypogaea(AH): 130 images
raw_data/philmedic/Philippine Medicinal Plant Leaf Dataset/8Pepromia pellucida(PP): 140 images
raw_data/philmedic/Philippine Medici

Merge Datasets (>5,000 Original Records) & Split Data

In [ ]:
import os
import shutil
import random
import pandas as pd
from pathlib import Path

random.seed(42)

# Adjust these two paths if Kaggle unzipped into an inner folder
PHILMEDIC_DIR = Path("raw_data/philmedic")
PLANTVILLAGE_DIR = Path("raw_data/plantvillage/plantvillage dataset/color")

PROCESSED_DIR = Path("processed_data")
if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

splits = ["train", "val", "test"]
for s in splits:
    (PROCESSED_DIR / s).mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / "ood_test").mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def get_images(folder):
    return [p for p in folder.iterdir() if p.suffix.lower() in VALID_EXTS]

summary_records = []

# 1. Process all Philippine Medicinal Plant classes (70/15/15 split)
medicinal_folders = [f for f in PHILMEDIC_DIR.rglob("*") if f.is_dir() and len(get_images(f)) > 0]

for class_dir in sorted(medicinal_folders):
    class_name = class_dir.name.strip().replace(" ", "_")
    imgs = get_images(class_dir)
    random.shuffle(imgs)

    n = len(imgs)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)

    split_map = {
        "train": imgs[:n_train],
        "val": imgs[n_train:n_train + n_val],
        "test": imgs[n_train + n_val:]
    }

    for split_name, file_list in split_map.items():
        dest = PROCESSED_DIR / split_name / class_name
        dest.mkdir(parents=True, exist_ok=True)
        for img_path in file_list:
            shutil.copy2(img_path, dest / img_path.name)

    summary_records.append({
        "Class": class_name,
        "Category": "Medicinal (Target)",
        "Train": len(split_map["train"]),
        "Val": len(split_map["val"]),
        "Test": len(split_map["test"]),
        "Total_Original": n
    })

# 2. Merge 1,500 Non-Medicinal PlantVillage images into 'Non_Medicinal_Unknown'
non_medicinal_sources = ["Tomato___healthy", "Potato___healthy", "Corn_(maize)___healthy"]
non_med_imgs = []
for src_name in non_medicinal_sources:
    src_path = PLANTVILLAGE_DIR / src_name
    if src_path.exists():
        imgs = get_images(src_path)
        random.shuffle(imgs)
        non_med_imgs.extend(imgs[:500])  # 500 per species = 1,500 original images

random.shuffle(non_med_imgs)
n_nm = len(non_med_imgs)
n_nm_train = int(0.70 * n_nm)
n_nm_val = int(0.15 * n_nm)

nm_splits = {
    "train": non_med_imgs[:n_nm_train],
    "val": non_med_imgs[n_nm_train:n_nm_train + n_nm_val],
    "test": non_med_imgs[n_nm_train + n_nm_val:]
}

for split_name, file_list in nm_splits.items():
    dest = PROCESSED_DIR / split_name / "Non_Medicinal_Unknown"
    dest.mkdir(parents=True, exist_ok=True)
    for idx, img_path in enumerate(file_list):
        shutil.copy2(img_path, dest / f"nonmed_{idx}_{img_path.name}")

summary_records.append({
    "Class": "Non_Medicinal_Unknown",
    "Category": "Non-Medicinal (Negative Class)",
    "Train": len(nm_splits["train"]),
    "Val": len(nm_splits["val"]),
    "Test": len(nm_splits["test"]),
    "Total_Original": n_nm
})

# 3. Hold out completely unseen species for OOD Rejection Evaluation
ood_sources = ["Apple___healthy", "Grape___healthy"]
ood_count = 0
for src_name in ood_sources:
    src_path = PLANTVILLAGE_DIR / src_name
    if src_path.exists():
        imgs = get_images(src_path)
        random.shuffle(imgs)
        for idx, img_path in enumerate(imgs[:150]):  # 300 unseen test images total
            shutil.copy2(img_path, PROCESSED_DIR / "ood_test" / f"ood_{src_name}_{idx}{img_path.suffix}")
            ood_count += 1

df_summary = pd.DataFrame(summary_records)
df_summary.to_csv("data_summary.csv", index=False)

total_original = df_summary["Total_Original"].sum()
print(f"Total Original Images in Train/Val/Test: {total_original} (Requirement: >5,000)")
print(f"Unseen OOD Test Images Held Out: {ood_count}")
display(df_summary.tail(10))

Total Original Images in Train/Val/Test: 6123 (Requirement: >5,000)
Unseen OOD Test Images Held Out: 300


,Class,Category,Train,Val,Test,Total_Original
31,39Manihot_esculenta(ME),Medicinal (Target),86,18,20,124
32,3Antidesma_bunius(AB),Medicinal (Target),99,21,22,142
33,40Citrus_maxima(CMA),Medicinal (Target),91,19,20,130
34,4Vitex_negundo(VN),Medicinal (Target),102,21,23,146
35,5Moringa_oleifera(MO),Medicinal (Target),96,20,22,138
36,6Blumea_balsamifera(BB),Medicinal (Target),77,16,17,110
37,7Origanum_vulgare(OV),Medicinal (Target),91,19,20,130
38,8Pepromia_pellucida(PP),Medicinal (Target),98,21,21,140
39,9Centella_asiatica(CA),Medicinal (Target),98,21,21,140
40,Non_Medicinal_Unknown,Non-Medicinal (Negative Class),806,172,174,1152


Train Baseline CNN vs. MobileNetV2 & Evaluate

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow Version:", tf.__version__)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 1. Load datasets from directory
train_ds = tf.keras.utils.image_dataset_from_directory(
    "processed_data/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", shuffle=True, seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    "processed_data/val", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    "processed_data/test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Loaded {num_classes} classes.")

# Save class index mapping for Streamlit app
class_indices = {str(i): name for i, name in enumerate(class_names)}
with open("class_indices.json", "w") as f:
    json.dump(class_indices, f, indent=2)

# 2. Compute balanced class weights to handle Non_Medicinal_Unknown (806 vs ~90 images)
y_train_int = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in train_ds], axis=0)
weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train_int), y=y_train_int)
class_weight_dict = dict(enumerate(weights))
print("Class weights computed successfully.")

# Optimize dataset performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# Training-only data augmentation layer
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
])

def evaluate_model(model, dataset, model_name):
    y_true = np.concatenate([y for x, y in dataset], axis=0)
    y_true_idx = np.argmax(y_true, axis=1)

    start_time = time.time()
    y_pred_probs = model.predict(dataset, verbose=0)
    inference_time = time.time() - start_time
    y_pred_idx = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y_true_idx, y_pred_idx)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true_idx, y_pred_idx, average="weighted", zero_division=0)
    cm = confusion_matrix(y_true_idx, y_pred_idx).tolist()

    print(f"\n--- {model_name} Test Results ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"Total Test Inference Time: {inference_time:.2f}s")

    return {
        "model": model_name,
        "accuracy": round(float(acc), 4),
        "precision_weighted": round(float(prec), 4),
        "recall_weighted": round(float(rec), 4),
        "f1_weighted": round(float(f1), 4),
        "inference_time_sec": round(float(inference_time), 2),
        "confusion_matrix": cm
    }

# ==========================================
# 3. BASELINE MODEL: Simple 3-Layer CNN
# ==========================================
baseline_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

baseline_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
print("\nTraining Baseline CNN...")
baseline_model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight_dict)
baseline_metrics = evaluate_model(baseline_model, test_ds, "Baseline_3Layer_CNN")

# ==========================================
# 4. PRIMARY MODEL: MobileNetV2 Transfer Learning
# ==========================================
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

mobilenet_model = models.Model(inputs, outputs)
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

print("\nTraining MobileNetV2 (Feature Extraction)...")
mobilenet_model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight_dict, callbacks=[early_stop])

# Fine-Tuning top 30 layers for higher accuracy
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("\nFine-Tuning Top Layers of MobileNetV2...")
mobilenet_model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weight_dict, callbacks=[early_stop])

mobilenet_metrics = evaluate_model(mobilenet_model, test_ds, "MobileNetV2_Transfer")

# Save primary model & evaluation report
mobilenet_model.save("leafcheck_mobilenetv2.keras")
with open("evaluation_report.json", "w") as f:
    json.dump({
        "class_names": class_names,
        "baseline": baseline_metrics,
        "primary": mobilenet_metrics
    }, f, indent=2)

print("\nSaved leafcheck_mobilenetv2.keras and evaluation_report.json!")

TensorFlow Version: 2.21.0
Found 4279 files belonging to 41 classes.
Found 903 files belonging to 41 classes.
Found 941 files belonging to 41 classes.
Loaded 41 classes.
Class weights computed successfully.

Training Baseline CNN...
Epoch 1/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 357s 3s/step - accuracy: 0.0559 - loss: 3.5327 - val_accuracy: 0.1894 - val_loss: 2.9272
Epoch 2/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 236s 2s/step - accuracy: 0.2358 - loss: 2.9003 - val_accuracy: 0.2060 - val_loss: 2.6261
Epoch 3/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 231s 2s/step - accuracy: 0.2718 - loss: 2.6905 - val_accuracy: 0.2458 - val_loss: 2.4718
Epoch 4/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 232s 2s/step - accuracy: 0.3115 - loss: 2.5159 - val_accuracy: 0.3134 - val_loss: 2.2488
Epoch 5/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 228s 2s/step - accuracy: 0.3494 - loss: 2.4169 - val_accuracy: 0.3677 - val_loss: 2.0046
Epoch 6/10
134/134 ━━━━━━━━━━━━━━━━━━━━ 231s 2s/step - accuracy: 0.3872 - loss: 2.2884 - val_accuracy: 0.3798 - val_loss: 

Evaluate OOD Rejection Threshold & Package Artifacts for VS Code

In [ ]:
import os
import json
import glob
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from google.colab import files

IMG_SIZE = (224, 224)

# Reload model and class mappings from disk in case runtime was idle
mobilenet_model = tf.keras.models.load_model("leafcheck_mobilenetv2.keras")
with open("class_indices.json", "r") as f:
    class_indices = json.load(f)
class_names = [class_indices[str(i)] for i in range(len(class_indices))]

# 1. Load unseen OOD images (Apple & Grape healthy leaves)
ood_paths = glob.glob("processed_data/ood_test/*")
ood_imgs = np.array([np.array(Image.open(p).convert("RGB").resize(IMG_SIZE)) for p in ood_paths])
ood_preds = mobilenet_model.predict(ood_imgs, verbose=0)
ood_max_conf = np.max(ood_preds, axis=1)
ood_pred_classes = [class_names[i] for i in np.argmax(ood_preds, axis=1)]

# 2. Load known medicinal test images (excluding Non_Medicinal_Unknown)
test_ds = tf.keras.utils.image_dataset_from_directory(
    "processed_data/test", image_size=IMG_SIZE, batch_size=32, label_mode="categorical", shuffle=False
)
non_med_idx = class_names.index("Non_Medicinal_Unknown")

known_imgs = []
for x_batch, y_batch in test_ds:
    for img, lbl in zip(x_batch.numpy(), y_batch.numpy()):
        if np.argmax(lbl) != non_med_idx:
            known_imgs.append(img)

known_imgs = np.array(known_imgs)
known_preds = mobilenet_model.predict(known_imgs, verbose=0)
known_max_conf = np.max(known_preds, axis=1)
known_pred_idx = np.argmax(known_preds, axis=1)

# 3. Sweep confidence thresholds from 0.50 to 0.95
threshold_results = []
for thresh in np.arange(0.50, 0.96, 0.05):
    thresh = round(float(thresh), 2)
    # OOD plant is falsely accepted ONLY if predicted as a medicinal class AND conf >= thresh
    false_accepts = sum(
        (pred_cls != "Non_Medicinal_Unknown") and (conf >= thresh)
        for pred_cls, conf in zip(ood_pred_classes, ood_max_conf)
    )
    far = false_accepts / len(ood_paths)

    # Known medicinal plant is properly accepted if predicted != Non_Medicinal_Unknown AND conf >= thresh
    true_accepts = sum(
        (pred_i != non_med_idx) and (conf >= thresh)
        for pred_i, conf in zip(known_pred_idx, known_max_conf)
    )
    tar = true_accepts / len(known_imgs)

    threshold_results.append({
        "threshold": thresh,
        "OOD_False_Acceptance_Rate": round(far, 4),
        "OOD_Rejection_Rate": round(1.0 - far, 4),
        "Known_Medicinal_Acceptance_Rate": round(tar, 4)
    })

df_thresh = pd.DataFrame(threshold_results)
display(df_thresh)

with open("ood_threshold_report.json", "w") as f:
    json.dump(threshold_results, f, indent=2)

# 4. Bundle all artifacts into a single zip and download
artifacts = [
    "leafcheck_mobilenetv2.keras",
    "class_indices.json",
    "evaluation_report.json",
    "ood_threshold_report.json",
    "data_summary.csv"
]

with zipfile.ZipFile("leafcheck_artifacts.zip", "w") as zf:
    for file in artifacts:
        if os.path.exists(file):
            zf.write(file)

print("Downloading leafcheck_artifacts.zip...")
files.download("leafcheck_artifacts.zip")

Found 941 files belonging to 41 classes.


,threshold,OOD_False_Acceptance_Rate,OOD_Rejection_Rate,Known_Medicinal_Acceptance_Rate
0,0.50,0.0833,0.9167,0.9961
1,0.55,0.0767,0.9233,0.9883
2,0.60,0.0700,0.9300,0.9817
3,0.65,0.0667,0.9333,0.9739
4,0.70,0.0567,0.9433,0.9661
5,0.75,0.0500,0.9500,0.9609
6,0.80,0.0400,0.9600,0.9518
7,0.85,0.0200,0.9800,0.9361
8,0.90,0.0033,0.9967,0.9140
9,0.95,0.0000,1.0000,0.8735


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>